# AuthenticVision: Corrected Real-vs-AI Image Detector Training

This notebook fine-tunes a ConvNeXt-Tiny model on Tiny-GenImage. The previous model was a research holdout experiment that excluded Midjourney fakes; it was not a suitable default deployable checkpoint. This run uses all generator families present in the training split.

The validation and test sets use different images from the dataset's validation split, but the generator families overlap with training. This is an image-disjoint deployment evaluation, not a strict unseen-generator test. The model can still fail on modern or unrepresented generators, so use an external labelled test set for the actual image types you care about.

Expected dataset data size: about 8.36 GB according to the dataset card. License: CC-BY-NC-SA-4.0; this project is intended for non-commercial learning/research unless you obtain appropriate permissions.

Dataset card: https://huggingface.co/datasets/TheKernel01/Tiny-GenImage


## Step 0 — Choose a cloud GPU

In Colab, select **Runtime → Change runtime type → T4 GPU** (if available), save, and return here. Free GPU availability and session limits vary. Then run the cells from top to bottom. Do not try to train on your Core m3 laptop; use it later only for prediction.


In [ ]:
import os, sys, platform, torch
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU is active. Stop here and select a GPU runtime before training.')

## Step 1 — Choose where checkpoints are saved

The cell below attempts to mount Google Drive so checkpoints survive a Colab disconnect. If Google authentication fails, it falls back to temporary Colab storage. In the fallback case, download the results ZIP at the end before the runtime ends.


In [ ]:
from pathlib import Path
DRIVE_OUTPUT = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_OUTPUT = Path('/content/drive/MyDrive/AuthenticVision_Training/run_002')
    DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
    OUTPUT_IS_PERSISTENT = True
    print('Google Drive output folder:', DRIVE_OUTPUT)
except Exception as exc:
    print('Google Drive mount failed; using temporary Colab storage instead.')
    print('Reason:', type(exc).__name__, str(exc)[:300])
    DRIVE_OUTPUT = Path('/content/AuthenticVision_Training/run_002')
    DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
    OUTPUT_IS_PERSISTENT = False
print('Output folder:', DRIVE_OUTPUT)
print('Persistent Google Drive storage:', OUTPUT_IS_PERSISTENT)


## Step 2 — Upload the project ZIP

Download and extract nothing yet. Upload the ZIP you received from ChatGPT using the picker opened by this cell. The ZIP contains the training code, GUI, tests, and reports; it does not include the 8.36 GB dataset or model weights.

In [ ]:
from google.colab import files
import zipfile, shutil

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if not zip_names:
    raise RuntimeError('Please upload the AuthenticVision_GenImage_Trainer.zip file.')
project_zip = zip_names[0]
PROJECT_DIR = Path('/content/AuthenticVision_GenImage_Trainer')
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
extract_root = Path('/content/project_extract')
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(project_zip, 'r') as zf:
    zf.extractall(extract_root)
# The archive contains one top-level project folder.
folders = [p for p in extract_root.iterdir() if p.is_dir()]
if len(folders) == 1:
    source_dir = folders[0]
else:
    source_dir = extract_root
shutil.copytree(source_dir, PROJECT_DIR, dirs_exist_ok=True)
print('Project ready:', PROJECT_DIR)
print('Training script exists:', (PROJECT_DIR / 'train.py').exists())
if not (PROJECT_DIR / 'train.py').exists():
    raise FileNotFoundError('train.py not found in uploaded project ZIP.')

## Step 3 — Install the small dependencies

Colab already provides a CUDA-ready PyTorch/torchvision build. We intentionally do **not** reinstall torch so we don't accidentally remove GPU support.

In [ ]:
%pip -q install -r /content/AuthenticVision_GenImage_Trainer/requirements-cloud.txt

## Step 4 — Configure your first training run

This corrected run uses all available AI generator types during training, samples up to 2,000 AI images per generator, fine-tunes the backbone with a lower learning rate than the classification head, uses modest JPEG/compression augmentation, and runs up to 8 epochs. The best checkpoint is selected by validation AUC; validation loss is monitored and reduces learning rates when it plateaus.

The notebook saves `last_checkpoint.pt` after every epoch and the best validation-AUC checkpoint as `best_model.pt`. Use a fresh output folder `run_002`; do not resume the previous run because its model was trained with a different split and optimizer configuration.

In [ ]:
# Corrected deployment training configuration. Use a NEW output folder: run_002.
HOLDOUT_GENERATOR = 'all'  # Includes Midjourney + all other generator types present in train split.
EPOCHS = 8  # Maximum epochs; best validation-AUC model will be saved.
BATCH_SIZE = 16  # If GPU memory runs out, reduce to 8.
IMAGE_SIZE = 224
MAX_TRAIN_PER_GENERATOR = 2000
MAX_VAL_PER_CLASS = 1000
MAX_TEST_PER_CLASS = 1000
LEARNING_RATE = 0.0001  # classifier head
BACKBONE_LEARNING_RATE = 0.00001  # pretrained image encoder
TARGET_REAL_FALSE_POSITIVE_RATE = 0.05
NUM_WORKERS = 2
RESUME_FROM_LAST_CHECKPOINT = False  # Do NOT resume the old run_001/diagnostic_run checkpoint.

print('Training mode:', HOLDOUT_GENERATOR)
print('Output folder:', DRIVE_OUTPUT)
print('Max epochs:', EPOCHS)
print('Backbone LR:', BACKBONE_LEARNING_RATE, '| Head LR:', LEARNING_RATE)
print('The first download is several GB; internet must remain enabled.')


## Step 5 — Train and evaluate

This corrected run trains on **all generator families available in the dataset training split**, including Midjourney. Validation and test use disjoint images from the official validation split, but generator families overlap with training. This is an image-disjoint deployment test, not an unseen-generator test. The script uses balanced sampling, mild JPEG compression augmentation, label smoothing, differential learning rates (lower for the pretrained backbone), and lowers learning rates if validation loss plateaus.


In [ ]:
import subprocess, json, os, sys

os.environ.setdefault('HF_HOME', '/content/hf_cache')
command = [
    sys.executable, str(PROJECT_DIR / 'train.py'),
    '--output-dir', str(DRIVE_OUTPUT),
    '--holdout-generator', HOLDOUT_GENERATOR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--image-size', str(IMAGE_SIZE),
    '--max-train-per-generator', str(MAX_TRAIN_PER_GENERATOR),
    '--max-val-per-class', str(MAX_VAL_PER_CLASS),
    '--max-test-per-class', str(MAX_TEST_PER_CLASS),
    '--learning-rate', str(LEARNING_RATE),
    '--backbone-learning-rate', str(BACKBONE_LEARNING_RATE),
    '--target-fpr', str(TARGET_REAL_FALSE_POSITIVE_RATE),
    '--num-workers', str(NUM_WORKERS),
]
if not RESUME_FROM_LAST_CHECKPOINT:
    # Prevent stale checkpoints from a previous run from being mistaken for this model.
    import shutil
    for existing in list(DRIVE_OUTPUT.iterdir()):
        if existing.is_dir():
            shutil.rmtree(existing)
        else:
            existing.unlink()
    print('Cleared old run_002 files for a clean training run.')

resume_path = DRIVE_OUTPUT / 'last_checkpoint.pt'
if RESUME_FROM_LAST_CHECKPOINT:
    if not resume_path.exists():
        raise FileNotFoundError(f'No saved checkpoint exists to resume: {resume_path}')
    command += ['--resume', str(resume_path)]
print('Running command:', ' '.join(command))
subprocess.run(command, cwd=str(PROJECT_DIR), check=True)

## Step 6 — Read the evaluation results

Read `test_metrics`, including accuracy, ROC-AUC, false-positive rate and false-negative rate. This corrected deployable model has seen the listed generator families during training. For claims about entirely unseen generators, run a separate leave-one-generator-out experiment and do not deploy that checkpoint for the held-out generator. Always test a separate, labelled collection matching the images you actually expect to see.


In [ ]:
metrics_path = DRIVE_OUTPUT / 'metrics.json'
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    print(json.dumps(metrics, indent=2))
else:
    print('metrics.json not found yet. Check the training cell output for an error.')
print('Files saved:')
for path in sorted(DRIVE_OUTPUT.glob('*')):
    print(f'{path.name:35} {path.stat().st_size / (1024**2):8.1f} MB')

## Step 7 — If the runtime disconnects

If checkpoints were saved to Drive, reopen the notebook, repeat setup/upload cells, and set `RESUME_FROM_LAST_CHECKPOINT = True` while increasing `EPOCHS` to the target final epoch. Do not resume the old `run_001` or `diagnostic_run` checkpoint because the updated training mode and optimizer differ.


## Step 8 — Use your trained model locally

After the last notebook cell downloads `AuthenticVision_Training_Results_run_002.zip`, extract it and copy `best_model.pt` to the `outputs` folder of your Windows project. Run `scripts/setup_windows.bat`, then `scripts/run_app_windows.bat`. For a quick command-line test:

```powershell
python predict.py --checkpoint outputs/best_model.pt --image "D:\photos\test.jpg"
```

The model is an educational detector, not forensic proof. Keep independently labelled real and AI images for external evaluation.


## Step 9 — Package results for Windows

Run the next cell after training. It downloads a ZIP containing the model and reports. If using temporary Colab storage, download it before disconnecting or the files will be lost.


In [ ]:
import shutil
from google.colab import files
archive_path = shutil.make_archive('/content/AuthenticVision_Training_Results_run_002', 'zip', root_dir=str(DRIVE_OUTPUT))
print('ZIP ready:', archive_path)
print('ZIP size MB:', round(Path(archive_path).stat().st_size / (1024**2), 1))
files.download(archive_path)
